# Relatório de Custos — Siplan

**Fluxo:** Power Automate → Fabric Notebook → PDF → SharePoint → Power Automate

O Power Automate injeta `atividade_ids_str` na célula de parâmetros abaixo.  
O notebook gera o PDF, salva no SharePoint e retorna o link via `mssparkutils.notebook.exit()`.

```
lake_gold_fatos.tabela_base   (1 linha por atividade — custos, sessões, autonomia)
lake_gold_fatos.contratos     (1 linha por solicitação — detalhe de itens de custo)
```

In [ ]:
# ── Parâmetros injetados pelo Power Automate ──────────────────────────────────
# Power Automate injeta atividade_ids_str como string CSV ou JSON array.
# Os demais valores podem ser fixos ou também injetados conforme necessidade.

atividade_ids_str = "63000016479883,63000018896693,63000018461299"

# URL do site SharePoint (sem barra final)
SHAREPOINT_SITE_URL = "https://sescsp.sharepoint.com/sites/GTDadosSTS"

# Pasta dentro do Documents (biblioteca padrão)
SHAREPOINT_FOLDER = "/Shared Documents/General/UO_para_STS"

# Nome da biblioteca de documentos (normalmente 'Documents' ou 'Documentos')
SHAREPOINT_LIBRARY = "Documents"

In [ ]:
import json
import os
import re
import tempfile
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import FancyBboxPatch
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')

# ── Estilo global ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
})

# ── Paleta de cores ───────────────────────────────────────────────────────────
AZUL   = '#1464A5'
VERDE  = '#2E8B57'
LARANJA= '#E07B39'
ROXO   = '#6B4C9A'
CINZA  = '#6C757D'
VERMELHO = '#C62828'

COR_AUTONOMIA = {'DIREG': VERMELHO, 'STS': LARANJA, 'UO': VERDE}

COR_TIPO_CUSTO = {
    'Contrato PF':      '#1464A5',
    'Contrato PJ':      '#1976D2',
    'Contrato Cooperativa': '#42A5F5',
    'Passagem Aérea':   '#E07B39',
    'Hospedagem':       '#8BC34A',
    'Alimentação':      '#FFB300',
    'Sonorização':      '#AB47BC',
    'Audiovisual':      '#7B1FA2',
    'Locação':          '#546E7A',
    'Comunicação':      '#00897B',
    'Outros':           '#90A4AE',
}

def fmt_brl(valor):
    if pd.isna(valor) or valor == 0:
        return '—'
    return 'R$ ' + f'{valor:,.0f}'.replace(',', '.')

def fmt_brl_k(valor, pos=None):
    if pd.isna(valor) or valor == 0:
        return '0'
    if valor >= 1000:
        return f'R$ {valor/1000:.0f}k'
    return f'R$ {valor:.0f}'

print('Imports OK — ', datetime.now().strftime('%d/%m/%Y %H:%M'))

In [ ]:
# ── Detecção de ambiente (Fabric vs. local) ───────────────────────────────────
try:
    spark
    FABRIC_ENV = True
    print('Ambiente: Microsoft Fabric')
except NameError:
    FABRIC_ENV = False
    print('Ambiente: local (modo simulação)')

try:
    from notebookutils import mssparkutils as _ms
    mssparkutils = _ms
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

# ── Parse dos IDs ─────────────────────────────────────────────────────────────
# Aceita: "id1,id2" ou '["id1","id2"]' (Power Automate envia JSON às vezes)
raw = atividade_ids_str.strip()
if raw.startswith('['):
    atividade_ids = json.loads(raw)
else:
    atividade_ids = [x.strip() for x in raw.split(',') if x.strip()]

print(f'Atividades recebidas ({len(atividade_ids)}): {atividade_ids}')

In [ ]:
# ── Dados de simulação (fallback quando lake_gold_fatos não está acessível) ───

def build_simulation_data(ids):
    """Retorna (df_base, df_contratos) com dados fictícios para os IDs informados."""

    base_rows = [
        {
            'atividade_id':           '63000016479883',
            'nome':                   'Curso de Música — Iniciação ao Violão',
            'servico':                'Curso',
            'subatividade':           'Ações formativas',
            'uo':                     63,
            'custo_contratos_total':  24800.0,
            'custo_total':            27350.0,
            'n_contratos':            2,
            'n_solic':                5,
            'qt_sessoes':             30,
            'qt_horas':               60.0,
            'estimativa_publico':     25,
            'lugares':                25,
            'por_sessao':             826.7,
            'por_hora':               413.3,
            'per_capita':             991.0,
            'autonomia':              'STS',
            'autonomiaCusto':         'STS-20',
            'autonomiaTemporal':      'STS',
            'periodicidade':          'permanente',
            'PrimeiraData':           '2025-03-01',
            'ultimadata':             '2025-11-30',
        },
        {
            'atividade_id':           '63000018896693',
            'nome':                   'Espetáculo de Teatro — Grupo Formação',
            'servico':                'Apresentação',
            'subatividade':           'Apresentação',
            'uo':                     63,
            'custo_contratos_total':  8500.0,
            'custo_total':            11200.0,
            'n_contratos':            1,
            'n_solic':                4,
            'qt_sessoes':             4,
            'qt_horas':               8.0,
            'estimativa_publico':     150,
            'lugares':                200,
            'por_sessao':             2125.0,
            'por_hora':               1062.5,
            'per_capita':             56.7,
            'autonomia':              'UO',
            'autonomiaCusto':         'UO',
            'autonomiaTemporal':      'UO',
            'periodicidade':          'eventual',
            'PrimeiraData':           '2025-06-10',
            'ultimadata':             '2025-06-13',
        },
        {
            'atividade_id':           '63000018461299',
            'nome':                   'Workshop de Produção Cultural Independente',
            'servico':                'Oficina',
            'subatividade':           'Ações formativas',
            'uo':                     63,
            'custo_contratos_total':  15200.0,
            'custo_total':            16800.0,
            'n_contratos':            1,
            'n_solic':                3,
            'qt_sessoes':             8,
            'qt_horas':               24.0,
            'estimativa_publico':     30,
            'lugares':                30,
            'por_sessao':             1900.0,
            'por_hora':               633.3,
            'per_capita':             506.7,
            'autonomia':              'STS',
            'autonomiaCusto':         'STS-15',
            'autonomiaTemporal':      'UO',
            'periodicidade':          'eventual',
            'PrimeiraData':           '2025-04-05',
            'ultimadata':             '2025-04-12',
        },
    ]

    contratos_rows = [
        # 63000016479883 — Curso de Violão
        {'atividade_id': '63000016479883', 'solicitacao_id': 'S001',
         'grupo': 'Contrato PF', 'item': 'Prof. João Silva',
         'custo': 18000.0, 'complemento': 'Regência e teoria musical'},
        {'atividade_id': '63000016479883', 'solicitacao_id': 'S002',
         'grupo': 'Contrato PF', 'item': 'Prof. Ana Souza',
         'custo': 6800.0,  'complemento': 'Prática instrumental'},
        {'atividade_id': '63000016479883', 'solicitacao_id': 'S003',
         'grupo': 'Alimentação', 'item': 'Coffee break',
         'custo': 1200.0,  'complemento': 'Encerramento de módulo'},
        {'atividade_id': '63000016479883', 'solicitacao_id': 'S004',
         'grupo': 'Comunicação', 'item': 'Impressos e digitais',
         'custo': 950.0,   'complemento': 'Material didático'},
        {'atividade_id': '63000016479883', 'solicitacao_id': 'S005',
         'grupo': 'Locação', 'item': 'Instrumentos',
         'custo': 400.0,   'complemento': 'Violões para o curso'},

        # 63000018896693 — Teatro
        {'atividade_id': '63000018896693', 'solicitacao_id': 'S006',
         'grupo': 'Contrato PJ', 'item': 'Cia Teatro Formação',
         'custo': 8500.0,  'complemento': 'Espetáculo + montagem'},
        {'atividade_id': '63000018896693', 'solicitacao_id': 'S007',
         'grupo': 'Passagem Aérea', 'item': 'Passagem POA→SP (4 artistas)',
         'custo': 1800.0,  'complemento': 'Ida e volta'},
        {'atividade_id': '63000018896693', 'solicitacao_id': 'S008',
         'grupo': 'Hospedagem', 'item': 'Hotel Centro — 2 noites',
         'custo': 900.0,   'complemento': '4 quartos duplos'},

        # 63000018461299 — Workshop
        {'atividade_id': '63000018461299', 'solicitacao_id': 'S009',
         'grupo': 'Contrato PF', 'item': 'Profa. Maria Costa',
         'custo': 15200.0, 'complemento': 'Facilitação e material'},
        {'atividade_id': '63000018461299', 'solicitacao_id': 'S010',
         'grupo': 'Alimentação', 'item': 'Kit Lanche — 8 encontros',
         'custo': 1600.0,  'complemento': '30 participantes × 8 sessões'},
    ]

    df_base = pd.DataFrame([r for r in base_rows if r['atividade_id'] in ids])
    df_contratos = pd.DataFrame([r for r in contratos_rows if r['atividade_id'] in ids])

    return df_base, df_contratos

In [ ]:
# ── Carregamento de dados ────────────────────────────────────────────────────

COLUNAS_BASE = [
    'atividade_id', 'nome', 'servico', 'subatividade', 'uo',
    'custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic',
    'qt_sessoes', 'qt_horas', 'estimativa_publico', 'lugares',
    'autonomia', 'autonomiaCusto', 'autonomiaTemporal',
    'periodicidade', 'PrimeiraData', 'ultimadata',
]

COLUNAS_CONTRATOS = [
    'atividade_id', 'solicitacao_id', 'grupo', 'item',
    'custo', 'complemento',
]

def load_from_fabric(ids):
    ids_sql = "'" + "','".join(ids) + "'"

    # Tenta lake_gold_fatos primeiro, depois lake_gold_siplan como fallback
    for lakehouse in ['lake_gold_fatos', 'lake_gold_siplan']:
        try:
            df_b = (
                spark.table(f'{lakehouse}.tabela_base')
                .filter(f'atividade_id IN ({ids_sql})')
                .toPandas()
            )
            df_b = df_b[[c for c in COLUNAS_BASE if c in df_b.columns]]

            df_c = (
                spark.table(f'{lakehouse}.contratos')
                .filter(f'atividade_id IN ({ids_sql})')
                .toPandas()
            )
            df_c = df_c[[c for c in COLUNAS_CONTRATOS if c in df_c.columns]]

            if not df_b.empty:
                print(f'Carregado de {lakehouse}: {len(df_b)} atividade(s)')
                return df_b, df_c
        except Exception:
            continue

    raise RuntimeError('Nenhum lakehouse disponível')


MODO_SIMULACAO = False
if FABRIC_ENV:
    try:
        df_base, df_contratos = load_from_fabric(atividade_ids)
    except Exception as err:
        print(f'[AVISO] Fabric indisponível ({err}). Usando dados de simulação.')
        df_base, df_contratos = build_simulation_data(atividade_ids)
        MODO_SIMULACAO = True
else:
    print('[SIMULAÇÃO] Ambiente local — usando dados fictícios para os 3 IDs de teste.')
    df_base, df_contratos = build_simulation_data(atividade_ids)
    MODO_SIMULACAO = True

# Garantir tipos numéricos
for col in ['custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic',
            'qt_sessoes', 'qt_horas', 'estimativa_publico', 'lugares']:
    if col in df_base.columns:
        df_base[col] = pd.to_numeric(df_base[col], errors='coerce').fillna(0)

if 'custo' in df_contratos.columns:
    df_contratos['custo'] = pd.to_numeric(df_contratos['custo'], errors='coerce').fillna(0)

print(f'df_base: {df_base.shape}')
print(f'df_contratos: {df_contratos.shape}')
display(df_base[['atividade_id', 'nome', 'custo_contratos_total', 'custo_total', 'autonomia']])

In [ ]:
# ── Preparação e métricas derivadas ─────────────────────────────────────────

# Público efetivo (lugares > estimativa_publico → usa lugares)
df_base['publico'] = df_base[['lugares', 'estimativa_publico']].max(axis=1).clip(lower=1)

# Métricas unitárias (calculadas a partir de custo_contratos_total)
df_base['por_sessao'] = np.where(
    df_base['qt_sessoes'] > 0,
    df_base['custo_contratos_total'] / df_base['qt_sessoes'], np.nan
)
df_base['por_hora'] = np.where(
    df_base['qt_horas'] > 0,
    df_base['custo_contratos_total'] / df_base['qt_horas'], np.nan
)
df_base['per_capita'] = np.where(
    df_base['publico'] > 0,
    df_base['custo_contratos_total'] / df_base['publico'], np.nan
)

# Nomes curtos para os gráficos (máx. 35 chars)
df_base['nome_curto'] = df_base['nome'].str[:35] + df_base['nome'].apply(
    lambda n: '…' if len(str(n)) > 35 else ''
)

# Custo total de contratos por atividade (para composição)
custo_por_tipo = (
    df_contratos
    .groupby(['atividade_id', 'grupo'])['custo']
    .sum()
    .reset_index()
)

# Ordem e agrupamento dos tipos de custo
ORDEM_GRUPO = [
    'Contrato PF', 'Contrato PJ', 'Contrato Cooperativa',
    'Passagem Aérea', 'Hospedagem', 'Alimentação',
    'Sonorização', 'Audiovisual', 'Locação', 'Comunicação', 'Outros',
]
custo_por_tipo['grupo'] = pd.Categorical(
    custo_por_tipo['grupo'],
    categories=ORDEM_GRUPO + [g for g in custo_por_tipo['grupo'].unique() if g not in ORDEM_GRUPO],
    ordered=True,
)
custo_por_tipo = custo_por_tipo.sort_values(['atividade_id', 'grupo'])

print('Métricas calculadas:')
print(df_base[['atividade_id', 'por_sessao', 'por_hora', 'per_capita', 'autonomia']].to_string(index=False))

In [ ]:
# ── Funções de geração de figuras ────────────────────────────────────────────

def fig_capa(df_base, data_geracao, modo_simulacao):
    fig = plt.figure(figsize=(11.69, 8.27))  # A4 landscape
    fig.patch.set_facecolor('#F0F4F8')
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()

    # Cabeçalho colorido
    ax.add_patch(FancyBboxPatch((0.02, 0.75), 0.96, 0.22,
                                boxstyle='round,pad=0.01',
                                facecolor=AZUL, edgecolor='none'))
    ax.text(0.50, 0.885, 'Relatório de Custos — Siplan',
            ha='center', va='center', fontsize=24, fontweight='bold', color='white')
    ax.text(0.50, 0.800, f'Gerado em {data_geracao}' + (' · DADOS DE SIMULAÇÃO' if modo_simulacao else ''),
            ha='center', va='center', fontsize=11, color='#BBDEFB')

    # Lista de atividades
    ax.text(0.05, 0.70, f'{len(df_base)} atividade(s) analisada(s):',
            fontsize=12, fontweight='bold', color='#333')

    y = 0.62
    for _, row in df_base.iterrows():
        cor = COR_AUTONOMIA.get(str(row.get('autonomia', '')), CINZA)
        ax.add_patch(FancyBboxPatch((0.05, y - 0.025), 0.90, 0.06,
                                    boxstyle='round,pad=0.005',
                                    facecolor='white', edgecolor='#DDD', linewidth=0.8))
        ax.text(0.08, y + 0.01, row['atividade_id'],
                fontsize=8, color=CINZA, va='center')
        ax.text(0.22, y + 0.01, str(row['nome'])[:55],
                fontsize=10, fontweight='bold', color='#222', va='center')
        ax.text(0.70, y + 0.01, f"{row.get('servico','')} · {row.get('subatividade','')[:20]}",
                fontsize=8.5, color='#555', va='center')
        ax.text(0.88, y + 0.01, fmt_brl(row.get('custo_total', 0)),
                fontsize=9, color=AZUL, va='center', ha='left', fontweight='bold')
        ax.add_patch(FancyBboxPatch((0.93, y - 0.012), 0.015, 0.036,
                                    boxstyle='round,pad=0.003',
                                    facecolor=cor, edgecolor='none'))
        y -= 0.085

    ax.text(0.50, 0.06, 'lake_gold_fatos · Fabric Notebook · Power Automate',
            ha='center', fontsize=8, color='#999')
    return fig


def fig_tabela_resumo(df_base):
    fig, ax = plt.subplots(figsize=(11.69, 8.27))
    ax.set_axis_off()
    fig.suptitle('Tabela Resumo de Custos', fontsize=14, fontweight='bold', y=0.97)

    colunas = ['ID', 'Nome', 'Serviço', 'Custo Total', 'Custo Contratos',
               'Sessões', 'Horas', 'R$/sessão', 'Per capita', 'Autonomia']

    def safe(row, col, fmt_fn=None):
        v = row.get(col, '—')
        if fmt_fn and not (pd.isna(v) if not isinstance(v, str) else v == ''):
            return fmt_fn(v)
        return str(v) if not pd.isna(v) else '—'

    dados = []
    for _, row in df_base.iterrows():
        dados.append([
            str(row['atividade_id'])[-6:],
            str(row['nome'])[:40],
            str(row.get('servico', '')),
            fmt_brl(row.get('custo_total', 0)),
            fmt_brl(row.get('custo_contratos_total', 0)),
            str(int(row.get('qt_sessoes', 0))),
            str(int(row.get('qt_horas', 0))),
            fmt_brl(row.get('por_sessao', np.nan)),
            fmt_brl(row.get('per_capita', np.nan)),
            str(row.get('autonomia', '')),
        ])

    tabela = ax.table(
        cellText=dados,
        colLabels=colunas,
        loc='center',
        cellLoc='center',
    )
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(8)
    tabela.scale(1, 2.0)

    # Estilo cabeçalho
    for j in range(len(colunas)):
        tabela[(0, j)].set_facecolor(AZUL)
        tabela[(0, j)].set_text_props(color='white', fontweight='bold')

    # Cores por autonomia (última coluna)
    for i, (_, row) in enumerate(df_base.iterrows(), start=1):
        cor_linha = '#F8F9FA' if i % 2 == 0 else 'white'
        for j in range(len(colunas)):
            tabela[(i, j)].set_facecolor(cor_linha)
        aut = str(row.get('autonomia', 'UO'))
        tabela[(i, len(colunas) - 1)].set_facecolor(COR_AUTONOMIA.get(aut, CINZA))
        tabela[(i, len(colunas) - 1)].set_text_props(color='white', fontweight='bold')

    plt.tight_layout()
    return fig


def fig_custo_total(df_base):
    fig, ax = plt.subplots(figsize=(11.69, 5))
    fig.suptitle('Custo Total e Custo de Contratos por Atividade', fontsize=13, fontweight='bold')

    y = np.arange(len(df_base))
    h = 0.35

    b1 = ax.barh(y + h/2, df_base['custo_total'],            h, label='Custo Total',      color=AZUL,   alpha=0.85)
    b2 = ax.barh(y - h/2, df_base['custo_contratos_total'],  h, label='Custo Contratos',  color=VERDE,  alpha=0.85)

    ax.set_yticks(y)
    ax.set_yticklabels(df_base['nome_curto'], fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl_k))
    ax.set_xlabel('Valor (R$)')
    ax.legend(loc='lower right')

    for bar in b1:
        w = bar.get_width()
        ax.text(w + w * 0.01, bar.get_y() + bar.get_height()/2,
                fmt_brl(w), va='center', fontsize=8, color=AZUL)
    for bar in b2:
        w = bar.get_width()
        ax.text(w + w * 0.01, bar.get_y() + bar.get_height()/2,
                fmt_brl(w), va='center', fontsize=8, color=VERDE)

    plt.tight_layout()
    return fig


def fig_composicao_custos(custo_por_tipo, df_base):
    tipos = custo_por_tipo['grupo'].unique()
    ativs = df_base['atividade_id'].tolist()
    nomes = df_base.set_index('atividade_id')['nome_curto']

    pivot = (
        custo_por_tipo
        .pivot_table(index='atividade_id', columns='grupo', values='custo', aggfunc='sum', fill_value=0)
        .reindex(ativs)
    )

    fig, ax = plt.subplots(figsize=(11.69, 5))
    fig.suptitle('Composição dos Custos por Tipo de Item', fontsize=13, fontweight='bold')

    bottom = np.zeros(len(ativs))
    for col in pivot.columns:
        vals = pivot[col].values
        cor = COR_TIPO_CUSTO.get(str(col), CINZA)
        bars = ax.bar(range(len(ativs)), vals, bottom=bottom, label=str(col), color=cor, width=0.5)
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 500:
                ax.text(i, b + v/2, fmt_brl_k(v), ha='center', va='center',
                        fontsize=7.5, color='white', fontweight='bold')
        bottom += vals

    ax.set_xticks(range(len(ativs)))
    ax.set_xticklabels([nomes.get(a, a) for a in ativs], fontsize=8.5)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl_k))
    ax.set_ylabel('Valor (R$)')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

    plt.tight_layout()
    return fig


def fig_metricas_unitarias(df_base):
    fig, axes = plt.subplots(1, 3, figsize=(11.69, 4.5))
    fig.suptitle('Métricas Unitárias de Custo', fontsize=13, fontweight='bold')

    metricas = [
        ('por_sessao',  'R$ por Sessão',   AZUL),
        ('por_hora',    'R$ por Hora',     VERDE),
        ('per_capita',  'Per Capita (Público Estimado)', LARANJA),
    ]

    for ax, (col, titulo, cor) in zip(axes, metricas):
        vals  = df_base[col].fillna(0).values
        names = df_base['nome_curto'].values
        bars = ax.bar(range(len(df_base)), vals, color=cor, alpha=0.85, width=0.55)
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_height() * 0.02,
                        fmt_brl(v), ha='center', va='bottom', fontsize=8)
        ax.set_xticks(range(len(df_base)))
        ax.set_xticklabels(names, fontsize=7, rotation=15, ha='right')
        ax.set_title(titulo, fontsize=10, fontweight='bold')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl_k))

    plt.tight_layout()
    return fig


def fig_detalhe_atividade(row_base, df_contratos_ativ):
    """Página detalhada por atividade: info + pizza + tabela de itens."""
    fig = plt.figure(figsize=(11.69, 8.27))
    gs  = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 1.8],
                             hspace=0.45, wspace=0.3,
                             left=0.05, right=0.95, top=0.90, bottom=0.05)

    fig.suptitle(f"{row_base['atividade_id']} · {row_base['nome']}",
                 fontsize=11, fontweight='bold', y=0.96)

    # ── Painel de informações (sup. esquerdo) ─────────────────────────────────
    ax_info = fig.add_subplot(gs[0, 0])
    ax_info.set_axis_off()

    infos = [
        ('Serviço',      f"{row_base.get('servico','')} · {row_base.get('subatividade','')}"),
        ('Período',      f"{row_base.get('PrimeiraData','')} → {row_base.get('ultimadata','')}"),
        ('Sessões / h',  f"{int(row_base.get('qt_sessoes',0))} sessões · {int(row_base.get('qt_horas',0))} h"),
        ('Público',      f"{int(row_base.get('publico',0))} pessoas"),
        ('Periodicidade',str(row_base.get('periodicidade',''))),
        ('Autonomia',    str(row_base.get('autonomia',''))),
        ('Custo Total',  fmt_brl(row_base.get('custo_total', 0))),
        ('Custo Contr.', fmt_brl(row_base.get('custo_contratos_total', 0))),
        ('R$/sessão',    fmt_brl(row_base.get('por_sessao', np.nan))),
        ('Per capita',   fmt_brl(row_base.get('per_capita', np.nan))),
    ]
    y0 = 0.98
    for label, valor in infos:
        ax_info.text(0.0, y0, f'{label}:', fontsize=8.5, fontweight='bold', color='#555', va='top')
        ax_info.text(0.38, y0, valor, fontsize=8.5, color='#222', va='top')
        y0 -= 0.097

    # ── Gráfico de pizza (sup. direito) ──────────────────────────────────────
    ax_pie = fig.add_subplot(gs[0, 1])
    if not df_contratos_ativ.empty:
        pie_data = df_contratos_ativ.groupby('grupo')['custo'].sum().sort_values(ascending=False)
        cores_pie = [COR_TIPO_CUSTO.get(str(g), CINZA) for g in pie_data.index]
        wedges, texts, autotexts = ax_pie.pie(
            pie_data.values, labels=None,
            autopct=lambda p: f'{p:.0f}%' if p > 5 else '',
            colors=cores_pie, startangle=90,
            pctdistance=0.75, wedgeprops={'linewidth': 0.8, 'edgecolor': 'white'},
        )
        for at in autotexts:
            at.set_fontsize(8)
            at.set_fontweight('bold')
        ax_pie.legend(pie_data.index, loc='lower center',
                      bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=7.5)
        ax_pie.set_title('Composição do Custo', fontsize=10, fontweight='bold')
    else:
        ax_pie.text(0.5, 0.5, 'Sem detalhamento\nde itens',
                    ha='center', va='center', fontsize=10, color=CINZA)
        ax_pie.set_axis_off()

    # ── Tabela de itens (linha inferior — ocupa as 2 colunas) ─────────────────
    ax_tab = fig.add_subplot(gs[1, :])
    ax_tab.set_axis_off()
    ax_tab.set_title('Detalhamento de Itens de Custo', fontsize=10,
                     fontweight='bold', loc='left', pad=8)

    if not df_contratos_ativ.empty:
        dados_tab = []
        for _, r in df_contratos_ativ.iterrows():
            dados_tab.append([
                str(r.get('grupo', '')),
                str(r.get('item', ''))[:40],
                str(r.get('complemento', ''))[:45],
                fmt_brl(r.get('custo', 0)),
            ])

        tb = ax_tab.table(
            cellText=dados_tab,
            colLabels=['Tipo', 'Item / Artista', 'Complemento', 'Custo'],
            loc='upper center',
            cellLoc='left',
            colWidths=[0.18, 0.32, 0.36, 0.14],
        )
        tb.auto_set_font_size(False)
        tb.set_fontsize(8.5)
        tb.scale(1, 1.8)
        for j in range(4):
            tb[(0, j)].set_facecolor(AZUL)
            tb[(0, j)].set_text_props(color='white', fontweight='bold')
        for i in range(1, len(dados_tab) + 1):
            cor = '#F8F9FA' if i % 2 == 0 else 'white'
            for j in range(4):
                tb[(i, j)].set_facecolor(cor)
    else:
        ax_tab.text(0.5, 0.5, 'Sem itens de custo cadastrados.',
                    ha='center', va='center', fontsize=10, color=CINZA)

    return fig


print('Funções de gráfico definidas.')

In [ ]:
# ── Geração do PDF ────────────────────────────────────────────────────────────

data_geracao = datetime.now().strftime('%d/%m/%Y %H:%M')
timestamp    = datetime.now().strftime('%Y%m%d_%H%M%S')
pdf_filename = f'relatorio_custos_{timestamp}.pdf'

# Em Fabric, /tmp está disponível; em produção pode salvar em /lakehouse/default/Files/
if FABRIC_ENV:
    pdf_dir = Path('/lakehouse/default/Files/relatorios')
    pdf_dir.mkdir(parents=True, exist_ok=True)
else:
    pdf_dir = Path(tempfile.gettempdir())

pdf_path = pdf_dir / pdf_filename

with PdfPages(str(pdf_path)) as pdf:

    # Página 1 — Capa
    f = fig_capa(df_base, data_geracao, MODO_SIMULACAO)
    pdf.savefig(f, bbox_inches='tight')
    plt.close(f)

    # Página 2 — Tabela Resumo
    f = fig_tabela_resumo(df_base)
    pdf.savefig(f, bbox_inches='tight')
    plt.close(f)

    # Página 3 — Custo Total vs. Contratos
    f = fig_custo_total(df_base)
    pdf.savefig(f, bbox_inches='tight')
    plt.close(f)

    # Página 4 — Composição dos Custos
    if not custo_por_tipo.empty:
        f = fig_composicao_custos(custo_por_tipo, df_base)
        pdf.savefig(f, bbox_inches='tight')
        plt.close(f)

    # Página 5 — Métricas Unitárias
    f = fig_metricas_unitarias(df_base)
    pdf.savefig(f, bbox_inches='tight')
    plt.close(f)

    # Páginas 6+ — Detalhamento por atividade
    for _, row in df_base.iterrows():
        aid = row['atividade_id']
        df_c_ativ = df_contratos[df_contratos['atividade_id'] == aid].copy()
        f = fig_detalhe_atividade(row, df_c_ativ)
        pdf.savefig(f, bbox_inches='tight')
        plt.close(f)

    # Metadados do PDF
    d = pdf.infodict()
    d['Title']   = 'Relatório de Custos Siplan'
    d['Author']  = 'Fabric Notebook — Power Automate'
    d['Subject'] = f"Atividades: {', '.join(atividade_ids)}"

print(f'PDF gerado: {pdf_path}')
print(f'Tamanho: {pdf_path.stat().st_size / 1024:.1f} KB')

In [ ]:
# ── Upload para o SharePoint via Microsoft Graph API ─────────────────────────
#
# Pré-requisito: a identidade do Fabric Notebook (ou SP configurado no Key Vault)
# precisa ter permissão Files.ReadWrite no site SharePoint alvo.
#
# Para descobrir site_id e drive_id execute uma vez:
#   GET https://graph.microsoft.com/v1.0/sites/{hostname}:/{site-path}
#   GET https://graph.microsoft.com/v1.0/sites/{site-id}/drives

import requests

SHAREPOINT_URL = ''

if not HAS_MSSPARKUTILS:
    print('[SIMULAÇÃO] mssparkutils não disponível — upload pulado.')
    SHAREPOINT_URL = f'<simulação> {pdf_filename}'
elif not SHAREPOINT_SITE_URL or SHAREPOINT_SITE_URL.startswith('https://seudominio'):
    print('[AVISO] SHAREPOINT_SITE_URL não configurado — upload pulado.')
    SHAREPOINT_URL = f'(configure SHAREPOINT_SITE_URL) — arquivo: {pdf_path}'
else:
    try:
        # Token para Graph API
        token = mssparkutils.credentials.getToken('https://graph.microsoft.com')
        headers_auth = {
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
        }

        # 1. Resolve site_id a partir da URL do site
        from urllib.parse import urlparse
        parsed = urlparse(SHAREPOINT_SITE_URL)
        hostname = parsed.netloc                       # ex: sesc.sharepoint.com
        site_path = parsed.path.lstrip('/')            # ex: sites/siplan-relatorios

        r = requests.get(
            f'https://graph.microsoft.com/v1.0/sites/{hostname}:/{site_path}',
            headers=headers_auth, timeout=30
        )
        r.raise_for_status()
        site_id = r.json()['id']

        # 2. Resolve drive_id da biblioteca padrão (ou pelo nome da biblioteca)
        r2 = requests.get(
            f'https://graph.microsoft.com/v1.0/sites/{site_id}/drives',
            headers=headers_auth, timeout=30
        )
        r2.raise_for_status()
        drives = r2.json()['value']
        drive = next(
            (d for d in drives if d.get('name', '').lower() == SHAREPOINT_LIBRARY.lower()),
            drives[0]  # fallback: primeira biblioteca
        )
        drive_id = drive['id']

        # 3. Upload (até 4 MB usa PUT simples; acima usa upload session)
        pdf_bytes = pdf_path.read_bytes()
        remote_path = f'{SHAREPOINT_FOLDER}/{pdf_filename}'.lstrip('/')

        if len(pdf_bytes) < 4 * 1024 * 1024:
            r3 = requests.put(
                f'https://graph.microsoft.com/v1.0/drives/{drive_id}/root:/{remote_path}:/content',
                headers={**headers_auth, 'Content-Type': 'application/octet-stream'},
                data=pdf_bytes, timeout=120
            )
            r3.raise_for_status()
            SHAREPOINT_URL = r3.json().get('webUrl', '')
        else:
            # Upload session para arquivos grandes
            r3 = requests.post(
                f'https://graph.microsoft.com/v1.0/drives/{drive_id}/root:/{remote_path}:/createUploadSession',
                headers=headers_auth,
                json={'item': {'@microsoft.graph.conflictBehavior': 'replace'}},
                timeout=30
            )
            r3.raise_for_status()
            upload_url = r3.json()['uploadUrl']
            chunk_size = 3 * 1024 * 1024
            for start in range(0, len(pdf_bytes), chunk_size):
                chunk = pdf_bytes[start:start + chunk_size]
                end   = start + len(chunk) - 1
                r_chunk = requests.put(
                    upload_url,
                    headers={'Content-Range': f'bytes {start}-{end}/{len(pdf_bytes)}',
                             'Content-Type': 'application/octet-stream'},
                    data=chunk, timeout=120
                )
            SHAREPOINT_URL = r_chunk.json().get('webUrl', upload_url)

        print(f'Upload concluído: {SHAREPOINT_URL}')

    except Exception as err:
        print(f'[ERRO] Upload falhou: {err}')
        SHAREPOINT_URL = f'ERRO_UPLOAD: {err}'

print(f'URL final: {SHAREPOINT_URL}')

In [ ]:
# ── Retorno para o Power Automate ─────────────────────────────────────────────
#
# O conector "Run a notebook" do Power Automate captura o valor passado para
# mssparkutils.notebook.exit() como string de saída da ação.

resultado = {
    'pdf_url':       SHAREPOINT_URL,
    'pdf_filename':  pdf_filename,
    'atividade_ids': atividade_ids,
    'total_atividades': len(df_base),
    'gerado_em':     datetime.now().isoformat(),
    'modo_simulacao': MODO_SIMULACAO,
}

resultado_json = json.dumps(resultado, ensure_ascii=False)

print('Retorno para o Power Automate:')
print(resultado_json)

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(resultado_json)
else:
    print('[LOCAL] mssparkutils.notebook.exit() não disponível — saída impressa acima.')